<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Lab 04 · The Hidden Costs of Portfolio Constraints

&copy; Dr. Yves J. Hilpisch<br>
AI-supported by GPT 5.x<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the lab examples in a Colab-ready format so that you can
run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the lab text for detailed explanations and context.


## Project Setup
Set the project root so local data files, helper modules, and figure scripts
resolve correctly from `notebooks/labs/`.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

loaded_code = sys.modules.get("code")
if loaded_code is not None and not hasattr(loaded_code, "__path__"):
    del sys.modules["code"]

PROJECT_ROOT

The lab uses a compact four-asset universe from `data/eod_data.csv`:
AAPL, NVDA, JPM, and SPY.


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv(
    "../../data/eod_data.csv",
    parse_dates=["Date"],
    index_col="Date",
)

In [ ]:
universe = ["AAPL", "NVDA", "JPM", "SPY"]

In [ ]:
prices = data[universe].dropna()

In [ ]:
returns = prices.pct_change().dropna().iloc[-2 * 252 :]

In [ ]:
returns[universe].head()

## Estimating Return and Risk Inputs
The first step is to estimate mean returns and covariance.


In [ ]:
import numpy as np

In [ ]:
mu_daily = returns[universe].mean()

In [ ]:
cov_daily = returns[universe].cov()

In [ ]:
mu_annual = (1.0 + mu_daily) ** 252 - 1.0

In [ ]:
cov_annual = cov_daily * 252.0

In [ ]:
diag_cov = np.diag(np.diag(cov_annual.values))

In [ ]:
lam = 0.25

In [ ]:
cov_shrunk = (
    lam * cov_annual.values + (1.0 - lam) * diag_cov
)

## Three Portfolio Versions
The lab compares three portfolios:


In [ ]:
Sigma = cov_shrunk

In [ ]:
mu_vec = mu_annual.values

In [ ]:
w_uncon = np.linalg.solve(Sigma, mu_vec)

In [ ]:
w_uncon = w_uncon / w_uncon.sum()

In [ ]:
w_long = np.clip(w_uncon, 0.0, None)

In [ ]:
w_long = w_long / w_long.sum()

In [ ]:
from code.labs.lab04_constraint_costs import build_portfolios

In [ ]:
weights, asset_rets, bench = build_portfolios()

In [ ]:
weights.round(3)

The next step is to summarize how these portfolios differ in expected
return, volatility, turnover from an equal-weight starting point, and
concentration.


In [ ]:
from code.labs.lab04_constraint_costs import summarize_tradeoffs

In [ ]:
summary = summarize_tradeoffs(weights)

In [ ]:
summary.round(3)

## Risk Contributions Can Shift Too
Even when the portfolio looks more balanced in weight terms, its risk
contributions may still be uneven.


In [ ]:
from code.labs.lab04_constraint_costs import estimate_moments

In [ ]:
from code.labs.lab04_constraint_costs import risk_contributions

In [ ]:
_, Sigma = estimate_moments(asset_rets)

In [ ]:
risk_contributions(
    weights["Asset + tech cap"],
    Sigma,
).round(3)

## Figure Generation (Optional)
Run the lab figure scripts under `code/figures/` to regenerate the PNG files
under `assets/figures/`.


In [ ]:
# Figure generation code adapted from
# `code/figures/lab04_constraint_tradeoffs.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = PROJECT_ROOT

from code.labs.lab04_constraint_costs import build_portfolios
from code.labs.lab04_constraint_costs import summarize_tradeoffs

def main() -> None:
    """Generate a four-panel trade-off figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    weights, _, _ = build_portfolios()
    stats = summarize_tradeoffs(weights).set_index("portfolio")

    fig, axes = plt.subplots(2, 2, figsize=(8.2, 5.4))
    cols = [
        ("exp_return", "Expected return"),
        ("volatility", "Volatility"),
        ("turnover", "Turnover from equal weight"),
        ("tracking_error", "Tracking error vs SPY"),
    ]
    colors = ["tab:blue", "tab:orange", "tab:green"]
    for ax, (col, title) in zip(axes.ravel(), cols):
        ax.bar(stats.index, stats[col], color=colors)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=12)
        ax.grid(True, axis="y", linestyle="--", alpha=0.3)

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab04_constraint_weights.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

ROOT = PROJECT_ROOT

from code.labs.lab04_constraint_costs import build_portfolios
from code.labs.lab04_constraint_costs import UNIVERSE

def main() -> None:
    """Generate a weight comparison figure for the lab."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    weights, _, _ = build_portfolios()
    fig, ax = plt.subplots(figsize=(7.2, 4.2))

    x = np.arange(len(UNIVERSE))
    width = 0.24
    ax.bar(
        x - width,
        weights["Unconstrained"],
        width,
        label="Unconstrained",
    )
    ax.bar(x, weights["Asset cap"], width, label="Asset cap")
    ax.bar(
        x + width,
        weights["Asset + tech cap"],
        width,
        label="Asset + tech cap",
    )

    ax.set_xticks(x)
    ax.set_xticklabels(UNIVERSE)
    ax.set_ylabel("Weight")
    ax.set_title("Weights under different constraint sets")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()

main()

In [ ]:
# Figure generation code adapted from
# `code/figures/lab04_risk_contributions.py`.

import os
import pathlib
import sys

os.environ["MPLCONFIGDIR"] = "/tmp/mplconfig"

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

ROOT = PROJECT_ROOT

from code.labs.lab04_constraint_costs import build_portfolios
from code.labs.lab04_constraint_costs import estimate_moments
from code.labs.lab04_constraint_costs import risk_contributions

def main() -> None:
    """Generate a risk-contribution comparison figure."""
    mpl.style.use("seaborn-v0_8")
    mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})

    weights, asset_rets, _ = build_portfolios()
    _, Sigma = estimate_moments(asset_rets)

    left = risk_contributions(weights["Asset cap"], Sigma)
    right = risk_contributions(weights["Asset + tech cap"], Sigma)
    data = np.column_stack([left.values, right.values])

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    x = np.arange(len(left.index))
    width = 0.34
    ax.bar(x - width / 2, data[:, 0], width, label="Asset cap")
    ax.bar(x + width / 2, data[:, 1], width, label="Asset + tech cap")
    ax.set_xticks(x)
    ax.set_xticklabels(left.index)
    ax.set_ylabel("Fraction of variance")
    ax.set_title("Risk contributions under different constraints")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    ax.legend(loc="best")

    fig.tight_layout()

main()

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
